In [1]:
import os
import numpy as np
import random
import nibabel as nib
from scipy.spatial.distance import euclidean
import bct
from functools import partial

import matplotlib.pyplot as plt

In [2]:
class Individual:
    def __init__(self, adj_matrix = None):
        self.adj_matrix = adj_matrix
        self.cost = None
        self.efficiency = None

def init_population(pop_size, brain_region_num, density):
    pop = []
    for _ in range(pop_size):
        adj_matrix = np.random.rand(brain_region_num, brain_region_num)
        off_diagonal = np.hstack([adj_matrix[np.triu_indices(brain_region_num, k = 1)], adj_matrix[np.tril_indices(brain_region_num, k = -1)]])
        thre = np.percentile(off_diagonal, int(density*100))
        adj_matrix = (adj_matrix < thre).astype(int)
        np.fill_diagonal(adj_matrix, 0)
        pop.append(Individual(adj_matrix))
    return pop

def calculate_objectives(population, cost_function, efficiency_function):
    for individual in population:
        individual.cost = cost_function(individual.adj_matrix)
        individual.efficiency = efficiency_function(individual.adj_matrix)
    return population

def cost(adj_matrix, distance): return np.sum(adj_matrix * distance)

def effi(adj_matrix): return bct.efficiency_bin(adj_matrix)

In [3]:
def crossover(parent1, parent2, num_points):
    size = parent1.shape[0]
    points = sorted(random.sample(range(size**2), num_points))
    child1 = parent1.copy().flatten()
    child2 = parent2.copy().flatten()
    for i in range(len(points) - 1):
        if i % 2 == 0:
            child1[points[i]:points[i+1]] = parent2.flatten()[points[i]:points[i+1]]
            child2[points[i]:points[i+1]] = parent1.flatten()[points[i]:points[i+1]]
    return child1.reshape(size, size), child2.reshape(size, size)

def mutate(individual, mutation_rate):
    adj_matrix = individual.adj_matrix.copy()
    size = adj_matrix.shape[0]
    rm = np.random.random((size, size)); np.fill_diagonal(rm, mutation_rate)
    adj_matrix[rm < mutation_rate] = 1 - adj_matrix[rm < mutation_rate]
    return adj_matrix

def guarantee(individual, count):
    adj_matrix = individual.adj_matrix.copy()
    diff = np.sum(adj_matrix) - count
    rows, cols = adj_matrix.shape
    indices = [(i, j) for i in range(rows) for j in range(cols) if i != j]
    if diff > 0:
        changed_count = 0
        while changed_count < diff:
            selected_indices = random.sample(indices, 1)[0]
            if adj_matrix[selected_indices[0], selected_indices[1]] == 1:
                adj_matrix[selected_indices[0], selected_indices[1]] = 0
                changed_count += 1
    if diff < 0:
        changed_count = 0
        while changed_count < abs(diff):
            selected_indices = random.sample(indices, 1)[0]
            if adj_matrix[selected_indices[0], selected_indices[1]] == 0:
                adj_matrix[selected_indices[0], selected_indices[1]] = 1
                changed_count += 1
    return adj_matrix

def dominates(p, q, reverse = False):
    if reverse == False: return (p.cost <= q.cost and p.efficiency >= q.efficiency) and (p.cost < q.cost or p.efficiency > q.efficiency)
    else: return (p.cost >= q.cost and p.efficiency <= q.efficiency) and (p.cost > q.cost or p.efficiency < q.efficiency)

def fast_non_dominated_sort(population, reverse = False):
    existed = [-1 for _ in population]
    for i, p in enumerate(population):
        if existed[i] == -1:
            existed[i] = i
            for j, q in enumerate(population):
                if existed[j] == -1 and approx(p, q): existed[j] = i
    fronts = [[]]
    for i, p in enumerate(population):
        p.domination_count = 0
        p.dominated_set = []
        for j, q in enumerate(population):
            if dominates(p, q, reverse):
                p.dominated_set.append(j)
            elif dominates(q, p, reverse):
                p.domination_count += 1
        if p.domination_count == 0:
            fronts[0].append(i)
    current_front = 0
    while fronts[current_front]:
        next_front = []
        for p_idx in fronts[current_front]:
            for q_idx in population[p_idx].dominated_set:
                population[q_idx].domination_count -= 1
                if population[q_idx].domination_count == 0:
                    next_front.append(q_idx)
        current_front += 1
        fronts.append(next_front)
    for idx in range(len(fronts)-1): fronts[idx] = list(set([existed[i] for i in fronts[idx]]))
    return fronts[:-1]

def approx(p, q):
    if abs(p.cost-q.cost)<0.1/200 and abs(p.efficiency-q.efficiency)<0.1/200: return True
    return False

In [4]:
def sample_list_evenly_with_ends(data, num_samples = 200):

    n = len(data)
    if n <= num_samples: return data
    step = (n - 1) / (num_samples - 1)
    sampled_indices = [round(step * i) for i in range(num_samples)]
    sampled_data = [data[i] for i in sampled_indices]
    
    return sampled_data

#### calculate Pareto (MMP)

In [5]:
atlas_gii_surf_L = nib.load('./inputs/S1200.L.pial_MSMAll.10k_fs_LR.surf.gii').darrays[0].data
atlas_gii_surf_R = nib.load('./inputs/S1200.R.pial_MSMAll.10k_fs_LR.surf.gii').darrays[0].data
atlas_gii_surf = np.vstack([atlas_gii_surf_L, atlas_gii_surf_R])
atlas_gii_label_L = nib.load('./inputs/L.Glasser2016.human.10k_fs_LR.label.gii').darrays[0].data
atlas_gii_label_R = nib.load('./inputs/R.Glasser2016.human.10k_fs_LR.label.gii').darrays[0].data
atlas_gii_label_R[atlas_gii_label_R != 0] += 1000
atlas_gii_label = np.hstack([atlas_gii_label_L, atlas_gii_label_R])
roi_labels = np.unique(atlas_gii_label); roi_labels = sorted(roi_labels[roi_labels != 0])
centroids = [np.mean(atlas_gii_surf[np.argwhere(atlas_gii_label == label).flatten()], axis = 0) for label in roi_labels]

distance = np.zeros((len(roi_labels), len(roi_labels)))
distance = np.array([[euclidean(centroids[i], centroids[j]) for j in range(len(roi_labels))] for i in range(len(roi_labels))])
distance /= np.sum(distance)

In [6]:
dist_flat = np.hstack([distance[np.triu_indices(distance.shape[0], k = 1)], distance[np.tril_indices(distance.shape[0], k = -1)]])
thre_list = [1e-10] + [np.percentile(dist_flat, 5*(i+1))+1e-10 for i in range(20)]
matr_list = [(distance < thre_list[i]).astype(int) - (distance < thre_list[i-3]).astype(int) for i in range(3, 20)]

In [ ]:
POP_SIZE = 200
MAX_GEN = 500
MUTATION_RATE = 0.01
CROSSOVER_POINTS = 10
BRAIN_REGION_NUM = 360
DENSITY = 0.15
CONNECT_COUNT = int(BRAIN_REGION_NUM*(BRAIN_REGION_NUM-1)*DENSITY)

In [8]:
os.makedirs('./outputs', exist_ok = True)
os.makedirs('./outputs/pareto_fig', exist_ok = True)

In [9]:
def nsga2(cost_function, efficiency_function):

    population = [Individual(matr) for matr in matr_list]
    population = calculate_objectives(population, cost_function, efficiency_function)

    fronts = fast_non_dominated_sort(population)

    for idx in fronts[0]: plt.scatter(population[idx].cost, population[idx].efficiency, color = 'black', s = 5, alpha = 0.5, linewidth = 0)
    plt.xlim(0.05, 0.20); plt.xticks([0.05, 0.10, 0.15, 0.20]); plt.xlabel('Cost')
    plt.ylim(0.4, 0.6); plt.yticks([0.4, 0.5, 0.6]); plt.ylabel('Efficiency')
    plt.text(0.06, 0.42, 'GENERATION: INITIAL')
    plt.savefig('./outputs/pareto_fig/GENERATION_INITIAL.jpg', format = 'jpg'); plt.close()

    for gen in range(MAX_GEN):

        offspring = []
        p1 = population[0]
        p2 = random.sample(population[1:], 1)[0]
        for i in range(4): offspring.append(Individual(guarantee(Individual(mutate(Individual(p1.adj_matrix), MUTATION_RATE*2/random.randint(1, 2**(i+4)))), CONNECT_COUNT)))
        child1, child2 = crossover(p1.adj_matrix, p2.adj_matrix, CROSSOVER_POINTS)
        offspring.append(Individual(guarantee(Individual(mutate(Individual(child1), MUTATION_RATE)), CONNECT_COUNT)))
        offspring.append(Individual(guarantee(Individual(mutate(Individual(child2), MUTATION_RATE)), CONNECT_COUNT)))
        while len(offspring) < POP_SIZE:
            p1, p2 = random.sample(population, 2)
            child1, child2 = crossover(p1.adj_matrix, p2.adj_matrix, CROSSOVER_POINTS)
            offspring.append(Individual(guarantee(Individual(mutate(Individual(child1), MUTATION_RATE)), CONNECT_COUNT)))
            offspring.append(Individual(guarantee(Individual(mutate(Individual(child2), MUTATION_RATE)), CONNECT_COUNT)))
        offspring = calculate_objectives(offspring, cost_function, efficiency_function)

        combined_population = population + offspring
        fronts = fast_non_dominated_sort(combined_population)

        population = []
        POPSIZE_OVERFLOW = False
        if len(fronts[0]) <= POP_SIZE: population.extend([combined_population[i] for i in fronts[0]])
        else:
            POPSIZE_OVERFLOW = True
            sorted_front = sorted(fronts[0], key=lambda x: combined_population[x].cost)
            sampled_sorted_front = sample_list_evenly_with_ends(sorted_front, POP_SIZE)
            population.extend([combined_population[i] for i in sampled_sorted_front])
        for front in fronts[1:]:
            if len(population) + len(front) <= POP_SIZE:
                population.extend([combined_population[i] for i in front])
            else:
                sorted_front = sorted(front, key=lambda x: combined_population[x].cost)
                population.extend([combined_population[i] for i in sorted_front[:POP_SIZE - len(population)]])
                break
        
        for idx in fronts[0]: plt.scatter(combined_population[idx].cost, combined_population[idx].efficiency, color = 'black', s = 5, alpha = 0.5, linewidth = 0)
        plt.xlim(0.05, 0.20); plt.xticks([0.05, 0.10, 0.15, 0.20]); plt.xlabel('Cost')
        plt.ylim(0.4, 0.6); plt.yticks([0.4, 0.5, 0.6]); plt.ylabel('Efficiency')
        plt.text(0.06, 0.42, 'GENERATION: {:05d}'.format(gen+1))
        plt.savefig('./outputs/pareto_fig/GENERATION_{:05d}.jpg'.format(gen+1), format = 'jpg'); plt.close()
        if POPSIZE_OVERFLOW: print('GENERATION: {:05d}, POPSIZE OVERFLOW!'.format(gen+1))

    return [combined_population[idx] for idx in fronts[0]]

In [10]:
pareto_population = nsga2(partial(cost, distance = distance), effi)

res = np.array([p.adj_matrix for p in pareto_population])
np.savez_compressed('./outputs/Pareto_MMP.npz', res)